# Fine-Tune Qwen 2.5 Coder 1.5B for Adobe After Effects ExtendScript
This notebook fine-tunes `Qwen2.5-Coder-1.5B-Instruct` on our comprehensive 3,181-sample After Effects ExtendScript dataset and exports a **quantized GGUF model (`q4_k_m`)** ready to drop into the AE extension with zero dependencies.

**Hardware:** Free Google Colab T4 GPU (Runtime -> Change runtime type -> T4 GPU).
**Training time:** ~15-20 minutes.

In [ ]:
# 1. Install Dependencies (Including unsloth_zoo)
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-cache-dir unsloth_zoo
!pip install --no-cache-dir trl peft accelerate bitsandbytes datasets transformers triton

In [ ]:
# 2. Load Base Model in 4-bit
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detect (Float16 or Bfloat16)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 3. Configure LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 4. Upload and Prepare Dataset
# Ensure 'ae_extendscript_train.jsonl' is in the Colab sidebar files!
from datasets import load_dataset

dataset = load_dataset("json", data_files="ae_extendscript_train.jsonl", split="train")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"Loaded {len(dataset)} training examples.")

In [ ]:
# 5. Train the Model
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 20,
        max_steps = 300, # ~2 epochs on dataset
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()
print("Training complete!")

In [ ]:
# 6. Test Model Inference
FastLanguageModel.for_inference(model)

prompt = "Create a 4K composition with a dark blue solid background and a centered text layer saying 'OFFLINE AI'"
messages = [
    {"role": "system", "content": "You are an expert Adobe After Effects ExtendScript assistant. You write safe, correct ExtendScript wrapped in undo groups."},
    {"role": "user", "content": prompt}
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=256, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# 7. Save and Export Directly to 4-bit GGUF (Q4_K_M)
model.save_pretrained_gguf("qwen2.5-coder-1.5b-ae", tokenizer, quantization_method = "q4_k_m")

# 8. Download the resulting GGUF file to your PC:
from google.colab import files
files.download("qwen2.5-coder-1.5b-ae/qwen2.5-coder-1.5b-ae-q4_k_m.gguf")